# 无标签小分子表示检查器

**模式：** 可编辑的学生练习。本 notebook 使用下方内嵌的七个作者编写教学示例。它不下载数据、不加载活性标签、不拟合模型、不定义数据划分，也不报告预测性能。

在 Colab 中，请下载此 `.ipynb` 文件，然后选择 **File > Upload notebook**。本地 Jupyter 也是替代方案。设置单元要求 Python 3.11 或更新版本，并使用 RDKit 2025.9.6。只有在缺失或版本不匹配时才安装该确切版本；若改变了 RDKit，重启 runtime 后再从头运行。本练习不需要 GPU。本 notebook 尚未在 Colab 中验证。

它会在 notebook 当前工作目录下新建目录并写入 `representation-report.json`，保留该文件供下载。它不会读取或改变作者结果、缓存或本地项目文件。不再需要时请自行删除生成的目录。重启 runtime 会清除变量；请按顺序重新运行各单元。

In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIRED_PYTHON = (3, 11)
REQUIRED_RDKIT = "2025.9.6"


def normalized_version(value):
    return tuple(int(part) for part in value.split("."))

if sys.version_info < REQUIRED_PYTHON:
    raise RuntimeError(
        f"Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]} or newer is required; "
        f"this runtime is {sys.version.split()[0]}."
    )

try:
    installed_rdkit = importlib.metadata.version("rdkit")
except importlib.metadata.PackageNotFoundError:
    installed_rdkit = None

loaded_rdkit = sys.modules.get("rdkit")
if loaded_rdkit is not None and normalized_version(
    getattr(loaded_rdkit, "__version__", "0")
) != normalized_version(REQUIRED_RDKIT):
    raise RuntimeError(
        "A different RDKit version is already loaded. Restart the runtime, then rerun "
        "this setup cell so it can install the pinned version safely."
    )

if installed_rdkit is None or normalized_version(installed_rdkit) != normalized_version(
    REQUIRED_RDKIT
):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", f"rdkit=={REQUIRED_RDKIT}"]
    )
    raise RuntimeError(
        "RDKit was installed or changed. Restart the runtime, then rerun this notebook "
        "from the first cell."
    )

import rdkit

if normalized_version(rdkit.__version__) != normalized_version(REQUIRED_RDKIT):
    raise RuntimeError(
        f"Loaded RDKit {rdkit.__version__}, expected {REQUIRED_RDKIT}. Restart the runtime."
    )

print(f"Python {sys.version.split()[0]}; RDKit {rdkit.__version__}")

## 问题与计算对象

SMILES 字符串是输入记法，不是实验测量。RDKit 将该记法解析为分子图，然后导出 canonical isomeric 和 non-isomeric SMILES、InChIKey、简单 2D descriptors、Murcko scaffold 以及两种 hashed Morgan fingerprint 摘要。代码报告二进制 fingerprint 的 occupied bit 数和 count fingerprint 的 nonzero feature 数。这两个数都不是生物学或理化 target。

下一段代码逐字嵌入仓库中经过测试的 inspection dataclass 和 function。将其保留在 notebook 中，可使上传后的文件独立运行，同时保持 canonical 的 radius 2、2,048-bit、chirality-aware Morgan configuration。

In [ ]:
from __future__ import annotations

import json
from collections.abc import Sequence
from dataclasses import asdict, dataclass

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold


class RepresentationInputError(ValueError):
    """Raised when a label-free representation input violates the lab contract."""


@dataclass(frozen=True)
class MoleculeInput:
    source_id: str
    smiles: str
    teaching_point: str


@dataclass(frozen=True)
class RepresentationConfig:
    morgan_radius: int = 2
    morgan_bits: int = 2048
    include_chirality: bool = True

    def __post_init__(self) -> None:
        if self.morgan_radius < 0:
            raise RepresentationInputError("Morgan radius must be non-negative")
        if self.morgan_bits <= 0:
            raise RepresentationInputError("Morgan bit-vector size must be positive")


@dataclass(frozen=True)
class DescriptorSnapshot:
    molecular_weight: float
    wildman_crippen_logp: float
    topological_polar_surface_area: float
    hydrogen_bond_donors: int
    hydrogen_bond_acceptors: int
    rotatable_bonds: int


@dataclass(frozen=True)
class MoleculeInspection:
    source_id: str
    input_smiles: str
    teaching_point: str
    canonical_isomeric_smiles: str
    canonical_nonisomeric_smiles: str
    inchi_key: str
    molecular_formula: str
    atom_count: int
    heavy_atom_count: int
    bond_count: int
    component_count: int
    formal_charge: int
    specified_stereocenters: int
    unassigned_stereocenters: int
    murcko_scaffold_smiles: str | None
    descriptors: DescriptorSnapshot
    morgan_on_bits: int
    morgan_nonzero_count_features: int


@dataclass(frozen=True)
class RepresentationContract:
    rdkit_version: str
    canonical_smiles_toolkit: str
    descriptor_names: tuple[str, ...]
    morgan_radius: int
    morgan_bits: int
    include_chirality: bool
    fingerprint_forms: tuple[str, ...]


@dataclass(frozen=True)
class RepresentationReport:
    scope: str
    contract: RepresentationContract
    molecules: tuple[MoleculeInspection, ...]


def inspect_molecules(
    inputs: Sequence[MoleculeInput],
    *,
    config: RepresentationConfig | None = None,
) -> tuple[MoleculeInspection, ...]:
    """Inspect molecular encodings without loading labels or fitting a model."""

    resolved_config = config or RepresentationConfig()
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=resolved_config.morgan_radius,
        fpSize=resolved_config.morgan_bits,
        includeChirality=resolved_config.include_chirality,
    )
    seen_ids: set[str] = set()
    inspections: list[MoleculeInspection] = []

    for record in inputs:
        if not record.source_id.strip():
            raise RepresentationInputError("source ID must not be empty")
        if record.source_id in seen_ids:
            raise RepresentationInputError(f"duplicate source ID: {record.source_id}")
        seen_ids.add(record.source_id)

        molecule = Chem.MolFromSmiles(record.smiles) if record.smiles.strip() else None
        if molecule is None or molecule.GetNumAtoms() == 0:
            raise RepresentationInputError(
                f"invalid or zero-atom SMILES for source ID {record.source_id}"
            )

        stereocenters = Chem.FindMolChiralCenters(  # type: ignore[no-untyped-call]
            molecule,
            force=True,
            includeUnassigned=True,
            includeCIP=True,
            useLegacyImplementation=False,
        )
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(  # type: ignore[no-untyped-call]
            mol=molecule
        )
        bit_fingerprint = generator.GetFingerprint(molecule)
        count_fingerprint = generator.GetCountFingerprint(molecule)
        inspections.append(
            MoleculeInspection(
                source_id=record.source_id,
                input_smiles=record.smiles,
                teaching_point=record.teaching_point,
                canonical_isomeric_smiles=Chem.MolToSmiles(
                    molecule, isomericSmiles=True
                ),
                canonical_nonisomeric_smiles=Chem.MolToSmiles(
                    molecule, isomericSmiles=False
                ),
                inchi_key=Chem.MolToInchiKey(molecule),  # type: ignore[no-untyped-call]
                molecular_formula=rdMolDescriptors.CalcMolFormula(molecule),
                atom_count=molecule.GetNumAtoms(),
                heavy_atom_count=molecule.GetNumHeavyAtoms(),
                bond_count=molecule.GetNumBonds(),
                component_count=len(Chem.GetMolFrags(molecule)),
                formal_charge=sum(
                    atom.GetFormalCharge()
                    for atom in molecule.GetAtoms()  # type: ignore[no-untyped-call]
                ),
                specified_stereocenters=sum(
                    assignment != "?" for _, assignment in stereocenters
                ),
                unassigned_stereocenters=sum(
                    assignment == "?" for _, assignment in stereocenters
                ),
                murcko_scaffold_smiles=scaffold or None,
                descriptors=DescriptorSnapshot(
                    molecular_weight=float(
                        Descriptors.MolWt(molecule)  # type: ignore[attr-defined]
                    ),
                    wildman_crippen_logp=float(
                        Descriptors.MolLogP(molecule)  # type: ignore[attr-defined]
                    ),
                    topological_polar_surface_area=float(
                        rdMolDescriptors.CalcTPSA(molecule)
                    ),
                    hydrogen_bond_donors=int(rdMolDescriptors.CalcNumHBD(molecule)),
                    hydrogen_bond_acceptors=int(rdMolDescriptors.CalcNumHBA(molecule)),
                    rotatable_bonds=int(
                        rdMolDescriptors.CalcNumRotatableBonds(molecule)
                    ),
                ),
                morgan_on_bits=len(bit_fingerprint.GetOnBits()),
                morgan_nonzero_count_features=len(
                    count_fingerprint.GetNonzeroElements()
                ),
            )
        )

    return tuple(inspections)


def build_report(
    inputs: Sequence[MoleculeInput],
    *,
    config: RepresentationConfig | None = None,
) -> RepresentationReport:
    resolved_config = config or RepresentationConfig()
    return RepresentationReport(
        scope="label-free representation inspection",
        contract=RepresentationContract(
            rdkit_version=rdBase.rdkitVersion,
            canonical_smiles_toolkit="RDKit",
            descriptor_names=(
                "molecular_weight",
                "wildman_crippen_logp",
                "topological_polar_surface_area",
                "hydrogen_bond_donors",
                "hydrogen_bond_acceptors",
                "rotatable_bonds",
            ),
            morgan_radius=resolved_config.morgan_radius,
            morgan_bits=resolved_config.morgan_bits,
            include_chirality=resolved_config.include_chirality,
            fingerprint_forms=("hashed_bit_vector", "hashed_count_vector"),
        ),
        molecules=inspect_molecules(inputs, config=resolved_config),
    )


def render_report(report: RepresentationReport) -> str:
    return json.dumps(asdict(report), indent=2, sort_keys=True) + "\n"

## 七个作者编写的示例

两条 pyridine 字符串在 RDKit 中应 canonicalize 为相同身份。两个已定义的 alanine 应保留不同的 isomeric 表示，但共享 non-isomeric 表示。未指定立体化学的 alanine 不会被分配给任一对映体。sodium acetate 保持为两个 component；本 notebook 不选择 parent fragment，也不进行 neutralization。

In [ ]:
EXAMPLES = (
    MoleculeInput(
        "pyridine-kekule",
        "C1=CC=CN=C1",
        "Equivalent graph written with explicit alternating bonds",
    ),
    MoleculeInput(
        "pyridine-aromatic",
        "n1ccccc1",
        "Equivalent graph written with aromatic atoms",
    ),
    MoleculeInput(
        "alanine-one",
        "N[C@@H](C)C(=O)O",
        "One defined tetrahedral stereoisomer",
    ),
    MoleculeInput(
        "alanine-two",
        "N[C@H](C)C(=O)O",
        "The opposite defined tetrahedral stereoisomer",
    ),
    MoleculeInput(
        "alanine-unspecified",
        "NC(C)C(=O)O",
        "Connectivity retained but tetrahedral stereochemistry unspecified",
    ),
    MoleculeInput("acetic-acid", "CC(=O)O", "Neutral protonation state"),
    MoleculeInput(
        "sodium-acetate",
        "CC(=O)[O-].[Na+]",
        "Disconnected salt with two components",
    ),
)
CONFIG = RepresentationConfig(
    morgan_radius=2,
    morgan_bits=2048,
    include_chirality=True,
)
report = build_report(EXAMPLES, config=CONFIG)
report_payload = json.loads(render_report(report))

assert len(report.molecules) == len(EXAMPLES)
assert report.contract.morgan_radius == 2
assert report.contract.morgan_bits == 2048
assert report.contract.include_chirality is True
print(f"Inspected {len(report.molecules)} label-free examples.")

## 检查、比较与绘图

表格比较由分子图导出的字段。SVG 图展示输入图，但不能建立溶液构象、结合 pose、affinity、efficacy、safety 或 synthesizability。canonicalization 同样不能决定 protonation、tautomer state 或 assay identity。

In [ ]:
from html import escape

from IPython.display import HTML, SVG, display
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D


def html_table(headers, rows):
    header_html = "".join(f"<th>{escape(str(header))}</th>" for header in headers)
    row_html = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    return (
        "<table><thead><tr>"
        + header_html
        + "</tr></thead><tbody>"
        + row_html
        + "</tbody></table>"
    )


def molecule_svg(smiles, legend):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise RepresentationInputError(f"could not draw invalid SMILES: {smiles}")
    drawer = rdMolDraw2D.MolDraw2DSVG(300, 220)
    rdMolDraw2D.PrepareAndDrawMolecule(drawer, molecule, legend=legend)
    drawer.FinishDrawing()
    return drawer.GetDrawingText()


by_id = {item.source_id: item for item in report.molecules}
display(
    HTML(
        html_table(
            ("example", "input", "canonical isomeric SMILES", "formula", "components"),
            [
                (
                    item.source_id,
                    item.input_smiles,
                    item.canonical_isomeric_smiles,
                    item.molecular_formula,
                    item.component_count,
                )
                for item in report.molecules
            ],
        )
    )
)

display(
    HTML(
        html_table(
            ("pyridine input", "canonical isomeric SMILES", "InChIKey"),
            [
                (item.input_smiles, item.canonical_isomeric_smiles, item.inchi_key)
                for item in (
                    by_id["pyridine-kekule"],
                    by_id["pyridine-aromatic"],
                )
            ],
        )
    )
)
display(
    HTML(
        html_table(
            (
                "alanine example",
                "isomeric SMILES",
                "non-isomeric SMILES",
                "specified / unassigned stereocenters",
            ),
            [
                (
                    item.source_id,
                    item.canonical_isomeric_smiles,
                    item.canonical_nonisomeric_smiles,
                    f"{item.specified_stereocenters} / {item.unassigned_stereocenters}",
                )
                for item in (
                    by_id["alanine-one"],
                    by_id["alanine-two"],
                    by_id["alanine-unspecified"],
                )
            ],
        )
    )
)
sodium_acetate = by_id["sodium-acetate"]
display(
    HTML(
        html_table(
            ("salt", "canonical isomeric SMILES", "components", "formal charge"),
            [
                (
                    sodium_acetate.source_id,
                    sodium_acetate.canonical_isomeric_smiles,
                    sodium_acetate.component_count,
                    sodium_acetate.formal_charge,
                )
            ],
        )
    )
)

for item in report.molecules:
    display(HTML(f"<h4>{escape(item.source_id)}</h4>"))
    display(SVG(molecule_svg(item.input_smiles, item.source_id)))

## 新鲜输出

输出单元会在当前工作目录下创建新的保留目录，以独占方式创建 `representation-report.json`，打印其路径并显示下载链接。不再需要时请自行删除该生成目录；不要把本练习指向冻结的作者结果目录。

In [ ]:
import tempfile
from pathlib import Path

from IPython.display import FileLink, display

output_directory = Path(
    tempfile.mkdtemp(prefix="aidd-representation-inspector-", dir=Path.cwd())
)
report_path = output_directory / "representation-report.json"
with report_path.open("x", encoding="utf-8") as report_file:
    report_file.write(render_report(report))
report_json = report_path.read_text(encoding="utf-8")

print(f"REPORT_PATH={report_path}")
print("Wrote a fresh JSON report and retained it for download.")
display(FileLink(str(report_path), result_html_prefix="Download: "))

print(
    "REPORT_JSON="
    + json.dumps(
        json.loads(report_json),
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    )
)

## 练习与答案

1. 在查看表格前，预测两条 pyridine 输入是否会有相同的 canonical isomeric SMILES 和 InChIKey。**答案：** 在此 RDKit 版本中会相同；它们编码同一个已解析分子图。
2. 哪个 alanine 字段丢失了两个已定义对映体之间的区别？**答案：** canonical non-isomeric SMILES；isomeric string 和 InChIKey 仍然不同。
3. 净电荷为零的 sodium acetate 在这里会变成一个中性分子吗？**答案：** 不会。即使 formal charge 总和为零，它仍是两个 component。
4. 在 `EXAMPLES` 中加入一个具有新 source ID 的有效 SMILES。预测哪些报告字段会改变。然后将它替换为 `not-a-smiles`。**答案：** 有效分子会增加一条额外的已检查记录；格式错误的输入会抛出 `RepresentationInputError`，而不是被静默排除。

这是一个表示学习练习，不是 QSAR、endpoint、training 或 evaluation protocol。